# Lecture 09 — Practice Exercises

All exercises from the Lecture 09 notebooks collected in one place.
Work through them after studying the corresponding lecture material.

| Section | Source Notebook | Topic |
|---|---|---|
| 1 | `lec_09a` | KMeans on the Mall Customers dataset |
| 2 | `lec_09a` | Choosing `K` — Elbow method and Silhouette score |
| 3 | `lec_09b` | KMeans assumptions and where it breaks |
| 4 | `lec_09e` / `lec_09d` | KMeans parameters and other clustering algorithms |


---
## Setup — Load Libraries and Data

Run this cell first to have everything ready for the exercises below.
The `mall_customers.csv` file lives in the `reading_material/` folder next door,
so we load it from there with a relative path.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


## 0. Enter the correct path to the dataset on your computer.

In [6]:
# Mall customers dataset (used in sections 1, 2 and 4)

DATA_PATH = '../../../data/mall_customers.csv'  # CHANGE THIS TO THE CORRECT PATH TO THE DATASET ON YOUR COMPUTER
df = pd.read_csv(DATA_PATH)


In [7]:

# Drop the useless ID column and rename the (Greek-letter-friendly) columns
df = df.drop(columns=["CustomerID"])

print(f"Mall customers dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)


Mall customers dataset: 200 rows, 4 columns


,Genre,Age,Annual_Income_(k$),Spending_Score
0,Male,19,15,39
1,Male,21,15,81
2,Female,20,16,6


---
## 1. KMeans on the Mall Customers dataset

*From `lec_09a_kmeans_clustering.ipynb`*

In the lecture we walked through the full KMeans workflow on this dataset:
load → EDA → fit → predict → evaluate → label clusters.
Now you do it yourself, with a slightly different setup each time.


### Exercise 1.1 — Quick EDA

Before fitting any model, get a feel for the data.

**Steps:**
1. Print the column names and data types.
2. Print descriptive statistics for the **numerical** columns.
3. Print the count (and percentage) of each value in `Genre`.
4. Build a `sns.pairplot()` of the numerical columns, coloured by `Genre`.

**Question to answer in a markdown cell below your code:**
Looking at the pairplot, between which two features do you visually see the
clearest cluster structure?


In [ ]:
# Exercise 1.1 — Your code here



### Exercise 1.2 — KMeans with two features and `K = 5`

Fit a `KMeans` model on the two features `Annual_Income_(k$)` and `Spending_Score`,
asking for **5 clusters**.

**Steps:**
1. Build the feature matrix `X` with these two columns.
2. Create `model = KMeans(n_clusters=5, random_state=42, n_init=10)`.
3. Use `fit_predict(X)` to get cluster labels in **one step**.
4. Add the labels as a new column `cluster` in `df`.
5. Plot the result as a 2D scatter (Income vs Spending), coloured by `cluster`,
   and **overlay the centroids** as black `X` markers.

**Hint:** the trained centroids are stored in `model.cluster_centers_`
(shape `(5, 2)`).


In [ ]:
# Exercise 1.2 — Your code here



### Exercise 1.3 — Predict the cluster of a new customer

Imagine three new customers walk into the mall:

| Customer | Annual income (k$) | Spending score |
|---|---|---|
| A | 20  | 80 |
| B | 75  | 50 |
| C | 110 | 15 |

Using the model you trained in Exercise 1.2, predict which cluster each one belongs to.

**Steps:**
1. Build a small `pandas.DataFrame` with the three new customers
   (use the same column names as `X` to avoid sklearn warnings).
2. Call `model.predict(...)` on it.
3. Print the predicted cluster for each customer.

**Why this exercise:** it is the cleanest way to see the difference between
`fit` (training, done once) and `predict` (using the trained model on
**new** observations).


In [ ]:
# Exercise 1.3 — Your code here



---
## 2. Choosing `K` — Elbow method and Silhouette score

*From `lec_09a_kmeans_clustering.ipynb`*

`K` is a hyperparameter you must pick yourself. We use two metrics in the lecture:
**WCSS** (within-cluster sum of squares, the "elbow") and the **silhouette score**.


### Exercise 2.1 — Elbow plot from K = 1 to K = 10

Compute the WCSS for `K = 1, 2, ..., 10` on the **two-feature** matrix
(`Annual_Income_(k$)`, `Spending_Score`) and plot it.

**Steps:**
1. Loop `k` from 1 to 10, fit a fresh `KMeans` for each, and append `model.inertia_`
   to a list.
2. Plot `K` on the x-axis and WCSS on the y-axis.
3. Mark with a vertical dashed line the `K` that you think is the elbow.

**Hint:** `inertia_` is the WCSS that scikit-learn computes for you.


In [ ]:
# Exercise 2.1 — Your code here



### Exercise 2.2 — Silhouette score from K = 2 to K = 10

The silhouette score is **not defined for K = 1** (you need at least two clusters).
Compute it for `K = 2, 3, ..., 10` on the same two features and plot it.

**Steps:**
1. Loop `k` from 2 to 10, fit `KMeans`, predict labels, and compute
   `silhouette_score(X, labels)`.
2. Plot `K` vs silhouette score.
3. Which `K` gives the **highest** score? Is it the same `K` the elbow
   suggested in Exercise 2.1?

**Hint:** higher silhouette is better (range is `[-1, 1]`).


In [ ]:
# Exercise 2.2 — Your code here



### Exercise 2.3 — Apply your chosen `K` with **four** features and profile the clusters

Now use **all four** features: `Genre`, `Age`, `Annual_Income_(k$)`, `Spending_Score`.

**Steps:**
1. Encode `Genre` as a numeric `Gender` column (`Male` → 0, `Female` → 1)
   so KMeans can use it.
2. Build `X` with the four numeric columns.
3. Fit `KMeans` with the `K` you chose in 2.1 / 2.2 (use `random_state=42`,
   `n_init=10`).
4. Add the labels as a `cluster` column on `df`.
5. Print the **mean of every numeric feature, grouped by cluster**.
6. In a markdown cell, give each cluster a short human label
   (e.g. *"young, low income, big spenders"*).

**Hint:** `df.groupby("cluster").mean(numeric_only=True)`


In [ ]:
# Exercise 2.3 — Your code here



---
## 3. KMeans assumptions and where it breaks

*From `lec_09b_kmeans_assumptions_caveats.ipynb`*

KMeans implicitly assumes that clusters are **roughly spherical**, **of similar
size**, and **of similar variance**. When those assumptions are violated, the
algorithm produces clusters that look obviously wrong to the human eye.
These exercises reproduce two of those failure modes on synthetic data.


### Exercise 3.1 — KMeans fails on non-convex shapes (`make_moons`)

Generate a small "two moons" dataset and try to cluster it with KMeans.

**Steps:**
1. `from sklearn.datasets import make_moons` and create
   `X, y = make_moons(n_samples=400, noise=0.06, random_state=42)`.
2. Plot `X` coloured by the true label `y` — you should see two interlocking moons.
3. Fit `KMeans(n_clusters=2, random_state=42, n_init=10)` on `X`, predict labels,
   and plot `X` coloured by **the KMeans labels** next to the true-label plot.

**Question to answer in a markdown cell below:**
Why does KMeans cut the moons across instead of along their shape?
What property of the data does it violate?


In [ ]:
# Exercise 3.1 — Your code here



### Exercise 3.2 — KMeans struggles with anisotropic (stretched) blobs

Same idea, different failure mode: clusters that are **stretched along
diagonals** instead of being round.

**Steps:**
1. Build three blobs with `make_blobs(n_samples=600, centers=3, cluster_std=0.6,
   random_state=42)`.
2. Apply an anisotropic linear transformation to make the blobs diagonal,
   e.g. `X_aniso = X @ np.array([[0.6, -0.6], [-0.4, 0.8]])`.
3. Fit `KMeans(n_clusters=3, random_state=42, n_init=10)` on `X_aniso`.
4. Plot the result coloured by predicted label and overlay the centroids.

**Question to answer in a markdown cell below:**
Compare your plot to the same data coloured by the **true** labels.
Where does KMeans "cut" the wrong way, and why?


In [ ]:
# Exercise 3.2 — Your code here



---
## 4. KMeans parameters and other clustering algorithms

*From `lec_09e_kmeans_other_parameters.ipynb` and `lec_09d_other_clustering_algos.ipynb`*

These exercises explore what changes when you tune KMeans' parameters,
and how a **density-based** algorithm (DBSCAN) compares on the same data.


### Exercise 4.1 — `init='random'` vs `init='k-means++'`

KMeans is sensitive to where the centroids start. By default scikit-learn
uses the smart initialisation `'k-means++'`, but you can ask for `'random'`.

**Steps:**
1. Use the **two-feature** mall-customers matrix (`Annual_Income_(k$)`,
   `Spending_Score`).
2. Fit two models with `K = 5`:
   - `KMeans(n_clusters=5, init="random", n_init=1, random_state=0)`
   - `KMeans(n_clusters=5, init="k-means++", n_init=1, random_state=0)`
3. Print `inertia_` for both. Which one is lower?
4. Repeat the experiment with `n_init=10`. Does the gap shrink?

**What this teaches:** why `n_init` exists and why `'k-means++'` is the default.


In [ ]:
# Exercise 4.1 — Your code here



### Exercise 4.2 — DBSCAN on the moons (where KMeans failed)

`DBSCAN` is a **density-based** algorithm: it does not need `K`, and it can
discover non-convex shapes. Re-run Exercise 3.1's `make_moons` data and
cluster it with DBSCAN.

**Steps:**
1. Recreate `X, y = make_moons(n_samples=400, noise=0.06, random_state=42)`.
2. `from sklearn.cluster import DBSCAN` and fit
   `DBSCAN(eps=0.2, min_samples=5)` on `X`.
3. Plot `X` coloured by the predicted label. Points DBSCAN considers
   **noise** receive label `-1`; show them in a distinct colour.
4. Try a few values of `eps` (e.g. 0.05, 0.2, 0.5) and observe what happens.

**Question to answer in a markdown cell below:**
For which `eps` value does DBSCAN recover the two moons cleanly?
What goes wrong when `eps` is too small or too large?


In [ ]:
# Exercise 4.2 — Your code here



---
## Summary

After completing all the exercises you will have practiced:

| Section | Skills practiced |
|---|---|
| 1 | Loading data, fitting `KMeans`, reading `cluster_centers_`, `predict()` on new points |
| 2 | Picking `K` with the elbow method and silhouette score, profiling clusters |
| 3 | Recognising when KMeans assumptions break (non-convex, anisotropic clusters) |
| 4 | Tuning `init` / `n_init`, comparing `KMeans` to `DBSCAN` |

**Key scikit-learn API patterns reinforced here:**
- `model.fit(X)` &nbsp;→&nbsp; train and store `cluster_centers_`, `labels_`, `inertia_`
- `model.predict(X_new)` &nbsp;→&nbsp; assign labels to **new** points
- `model.fit_predict(X)` &nbsp;→&nbsp; one-line shortcut for unsupervised learning
- `silhouette_score(X, labels)` &nbsp;→&nbsp; quality of a clustering
